# Google Vertex AI

**Vertex AI** is Google Cloud's unified, fully-managed platform for the whole ML lifecycle — training, tuning, deploying, and serving models — *and* the enterprise front door to Google's **Gemini** foundation models with IAM, VPC, and data-governance controls.

**Domain:** Proprietary Models & Coding AI  ·  **from study list**  ·  **runnable:** yes  ·  _needs GCP creds (live cells gate on `os.getenv`)_

## 1. What & Why

**Vertex AI** is Google Cloud's single, managed home for machine learning. Two faces matter:

1. **Generative AI front door** — the *enterprise* way to call Google's foundation models (Gemini 2.5 Pro/Flash, Imagen, embeddings, plus third-party "Model Garden" models like Anthropic's Claude). Same models as the consumer Gemini API, but wrapped in Google Cloud's auth, networking, and compliance.
2. **Classic ML platform** — managed training (custom + AutoML), a model registry, one-click **Endpoints** for online prediction, batch prediction, Pipelines, Feature Store, and experiment tracking.

**The problem it solves:** going from "I have a model or a prompt" to "it runs in production under our security and billing controls" without standing up your own serving infrastructure. You get autoscaling GPUs/TPUs, IAM-based access, audit logs, VPC Service Controls, and data residency — the things a regulated enterprise needs before it can ship.

**Reach for Vertex AI when:**
- You're **already on Google Cloud** and want billing, IAM, and logging unified with the rest of your stack.
- You need **enterprise governance** for LLM calls — VPC-SC, CMEK, data-not-used-for-training guarantees, regional pinning.
- You want **managed deployment** of a custom model (sklearn/PyTorch/TF) behind an autoscaling HTTPS endpoint without writing serving code.
- You want **one SDK** that reaches Gemini, Claude, Llama, and your own models through a common control plane.

**When NOT to:** for a quick prototype or a hobby project, the plain **Gemini API** (key from Google AI Studio) is far less ceremony — no GCP project, IAM, or billing setup. If you aren't on Google Cloud and don't need its governance, Vertex's overhead buys you little. For pure open-weight self-hosting with no cloud lock-in, use your own infra + a serving stack (vLLM, etc.).

## 2. Mental Model

Think of Vertex AI as **Google Cloud's managed ML control plane** — a project-scoped hub that turns models (yours or Google's) into governed, autoscaling endpoints:

```
            ┌─────────────────────────  Vertex AI (a GCP project + region)  ─────────────────────────┐
            │                                                                                          │
  Gen AI ──▶│  Foundation models:  Gemini 2.5 · Imagen · embeddings · Model Garden (Claude, Llama…)    │──▶ governed API call
            │        same models as the Gemini API, but behind IAM / VPC-SC / CMEK / audit logs         │
            │                                                                                          │
  Your  ───▶│  Training  ──▶  Model Registry  ──▶  Endpoint (autoscaling)  ──▶  online / batch predict  │──▶ HTTPS prediction
  model     │                                                                                          │
            └──────────────────────────────────────────────────────────────────────────────────────────┘
                         auth = Google Cloud IAM (service accounts / ADC),  not an API key
```

Three things to internalize:

1. **It's a control plane, not a model.** Vertex doesn't replace Gemini — it *governs your access* to Gemini and to your own models. Same Gemini weights as the consumer API; different door (IAM instead of an API key, a project + region instead of a global endpoint).
2. **Everything is project + region scoped.** A model deployed in `us-central1` of project `my-proj` is a distinct resource. Auth flows through **Application Default Credentials (ADC)** / service accounts, not a bearer key you paste in.
3. **Two lifecycles share one platform.** Calling Gemini ("just give me a completion") and deploying your own model ("train → register → endpoint → predict") are different workflows that live under the same SDK and console.

## 3. Key Concepts

- **Project & region** — every Vertex resource lives in a GCP **project** and a **region** (e.g. `us-central1`). Model availability and data residency are per-region. You always initialize with both.
- **Application Default Credentials (ADC)** — how you authenticate. `gcloud auth application-default login` for local dev, or an attached **service account** in production. No API keys — Vertex uses Google Cloud IAM.
- **`google-genai` SDK (Vertex mode)** — the *same* unified SDK as the Gemini API, switched to Vertex with `genai.Client(vertexai=True, project=..., location=...)` (or env vars). Your generation code is otherwise identical — that's the migration story.
- **Model Garden** — a catalog of 100+ models you can deploy/call through Vertex: Google's (Gemini, Imagen), partner models (**Anthropic Claude**, Meta **Llama**, Mistral), and open models. One billing + IAM surface for all of them.
- **Custom training** — run your own training code in managed containers on Vertex's GPUs/TPUs (CustomJob / training pipelines). **AutoML** is the no-code alternative for tabular/vision/text.
- **Model Registry** — versioned store of trained models. You register a model, then deploy a version to an endpoint.
- **Endpoint** — a managed, autoscaling HTTPS service for **online prediction** (low-latency, one request at a time). **Batch prediction** is the offline, high-throughput counterpart that reads/writes Cloud Storage or BigQuery.
- **Pipelines** — Vertex AI Pipelines (Kubeflow/TFX based) orchestrate multi-step ML workflows (preprocess → train → eval → deploy) as DAGs.
- **Grounding** — Vertex can ground Gemini answers in **Google Search** or your own data (Vertex AI Search / RAG Engine) for citations and reduced hallucination.
- **Governance knobs** — **VPC Service Controls** (network perimeter), **CMEK** (customer-managed encryption keys), regional pinning, and audit logging. The reasons you'd pick Vertex over the raw Gemini API.

## 4. Setup

Vertex needs a **Google Cloud project with billing** and the Vertex AI API enabled — there's no instant free API key like the consumer Gemini API.

```bash
# 1. Install the current unified SDK (same package as the Gemini API)
pip install google-genai            # for generative models
pip install google-cloud-aiplatform # for training / endpoints / registry

# 2. Authenticate with Application Default Credentials (local dev)
gcloud auth application-default login

# 3. Point the SDK at your project + region, in Vertex mode
export GOOGLE_GENAI_USE_VERTEXAI=true
export GOOGLE_CLOUD_PROJECT="your-project-id"
export GOOGLE_CLOUD_LOCATION="us-central1"

# 4. Enable the API once per project
gcloud services enable aiplatform.googleapis.com
```

A minimal Gemini-on-Vertex call is then almost identical to the consumer API — only the client construction differs:

```python
from google import genai
client = genai.Client(vertexai=True, project="your-project-id", location="us-central1")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Hello")
print(resp.text)
```

The cells below run top-to-bottom in a fresh kernel **without** the SDK or GCP creds — every live call is gated behind an `os.getenv` / credentials check.

In [ ]:
# This notebook executes with or without the SDK and GCP credentials.
# To run the live examples, uncomment the installs and authenticate first:
# %pip install google-genai google-cloud-aiplatform
# gcloud auth application-default login

import os

project  = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
use_vertex = os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1")

print("GOOGLE_CLOUD_PROJECT :", project or "(unset)")
print("GOOGLE_CLOUD_LOCATION:", location)
print("Vertex mode enabled  :", use_vertex)
print("Default model        :", "gemini-2.5-flash")
print("Auth model           : Application Default Credentials (no API key)")

## 5. Worked Examples

### Example 1 — Gemini API vs Vertex AI: the *only* code that differs (no network)

The migration story is "change the client, keep the call." Below we build both client configs side by side so you can see exactly what changes (and what doesn't) when you graduate from the consumer Gemini API to enterprise Vertex AI.

In [ ]:
# The same generation request, two front doors. Pure Python — no network, no creds.

# --- Consumer Gemini API: auth via an API key, global endpoint ---
gemini_api_client = {
    "constructor": "genai.Client()",
    "auth": "GEMINI_API_KEY environment variable",
    "scope": "global endpoint, no project/region",
}

# --- Vertex AI: auth via Google Cloud IAM (ADC), project + region scoped ---
vertex_client = {
    "constructor": "genai.Client(vertexai=True, project='my-proj', location='us-central1')",
    "auth": "Application Default Credentials / service account (IAM)",
    "scope": "project 'my-proj', region 'us-central1'",
}

# The generate_content call itself is IDENTICAL across both:
shared_call = "client.models.generate_content(model='gemini-2.5-flash', contents='Hello')"

print("WHAT CHANGES (client construction):")
for k in ("constructor", "auth", "scope"):
    print(f"  {k:12} Gemini API : {gemini_api_client[k]}")
    print(f"  {k:12} Vertex AI  : {vertex_client[k]}")
    print()
print("WHAT STAYS THE SAME (the actual request):")
print(" ", shared_call)

### Example 2 — Estimate token cost for a Vertex Gemini call (no network)

Vertex bills Gemini per token just like the consumer API (enterprise pricing/commitments may differ). A `chars / 4` heuristic is fine for back-of-envelope budgeting; use `client.models.count_tokens(...)` when precision matters.

In [ ]:
# Rough token + cost estimate for Gemini on Vertex. Pure Python, no API.
# Prices are ILLUSTRATIVE (USD per 1M tokens) — confirm at cloud.google.com/vertex-ai/pricing.
PRICING = {
    "gemini-2.5-flash":      {"in": 0.30, "out": 2.50},
    "gemini-2.5-pro":        {"in": 1.25, "out": 10.00},
    "gemini-2.5-flash-lite": {"in": 0.10, "out": 0.40},
}

def est_tokens(text: str) -> int:
    return max(1, len(text) // 4)        # ~4 chars/token for English prose

def est_cost(model: str, in_tokens: int, out_tokens: int) -> float:
    p = PRICING[model]
    return in_tokens / 1e6 * p["in"] + out_tokens / 1e6 * p["out"]

prompt = "Summarize our Q3 incident report and list the top 3 action items."
in_tok, out_tok = est_tokens(prompt), 120

print(f"prompt: {prompt!r}")
print(f"input ~{in_tok} tokens, output ~{out_tok} tokens\n")
for model in PRICING:
    print(f"  {model:<22} est ${est_cost(model, in_tok, out_tok):.6f}")

### Example 3 — The real Gemini-on-Vertex call, gated behind credentials

The actual SDK invocation in Vertex mode. With a project + ADC set it returns live text; without it, the cell prints the call shape so the notebook still executes cleanly end-to-end.

In [ ]:
import os

def call_vertex_gemini(prompt: str, model: str = "gemini-2.5-flash") -> str:
    from google import genai            # pip install google-genai
    from google.genai import types
    client = genai.Client(               # Vertex mode: IAM/ADC auth, project + region scoped
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1"),
    )
    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="Reply with a single short word.",
            temperature=0.0,
            max_output_tokens=16,
        ),
    )
    return resp.text

# Live call needs BOTH a project and working Application Default Credentials.
if os.getenv("GOOGLE_CLOUD_PROJECT"):
    try:
        print("Vertex Gemini says:", call_vertex_gemini("Reply with exactly: pong").strip())
    except Exception as e:               # missing ADC / quota / API not enabled
        print("Live call failed:", type(e).__name__, e)
else:
    print("GOOGLE_CLOUD_PROJECT unset — skipping the live call.")
    print("Call shape: genai.Client(vertexai=True, project=..., location=...)")
    print("            .models.generate_content(model='gemini-2.5-flash', contents=...)")

### Example 4 — Deploy a custom model: the platform workflow (call shape)

Beyond Gemini, Vertex's other half is deploying *your own* model behind an autoscaling endpoint. This is the canonical `aiplatform` sequence — register → deploy → predict. It's gated because a real endpoint costs money and takes minutes to spin up; the shape is what you want in muscle memory.

In [ ]:
import os

def deploy_and_predict(artifact_uri: str, instances: list):
    from google.cloud import aiplatform        # pip install google-cloud-aiplatform
    aiplatform.init(
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1"),
    )
    # 1. Register a trained model (artifact in GCS) into the Model Registry.
    model = aiplatform.Model.upload(
        display_name="my-sklearn-model",
        artifact_uri=artifact_uri,            # gs://bucket/path/to/model/
        serving_container_image_uri=(
            "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
        ),
    )
    # 2. Deploy to an autoscaling Endpoint (creates one if needed). Billed while up.
    endpoint = model.deploy(machine_type="n1-standard-2", min_replica_count=1)
    # 3. Online prediction.
    return endpoint.predict(instances=instances).predictions

# Real deployment provisions billable infra — only attempt with an explicit opt-in flag.
if os.getenv("GOOGLE_CLOUD_PROJECT") and os.getenv("VERTEX_RUN_DEPLOY") == "1":
    preds = deploy_and_predict("gs://my-bucket/model/", [[5.1, 3.5, 1.4, 0.2]])
    print("predictions:", preds)
else:
    print("Skipping deploy (set GOOGLE_CLOUD_PROJECT and VERTEX_RUN_DEPLOY=1 to run).")
    print("Workflow: Model.upload(...)  ->  model.deploy(...)  ->  endpoint.predict(...)")
    print("Remember to endpoint.undeploy_all() / endpoint.delete() to stop billing!")

## 6. Gotchas & Pitfalls

- **Endpoints bill while they exist, not per call.** A deployed model holds a reserved replica (a VM/GPU) running 24/7 — you pay for uptime even with zero traffic. Forgetting to `undeploy_all()` / `delete()` an experiment endpoint is the classic surprise bill. Use **batch prediction** for offline jobs, and set `min_replica_count=0` only where scale-to-zero is supported.
- **Auth is IAM, not an API key.** "It works on my laptop, 403 in prod" almost always means ADC vs service-account confusion. Locally you have *your* `gcloud` creds; in prod the workload's **service account** needs the `roles/aiplatform.user` (and often `storage.objectViewer`) role. There's no key to paste.
- **Region matters — for availability *and* residency.** Not every model is in every region, and a resource in `us-central1` can't be called as if it were global. Pin region deliberately; for data residency, keep prompts and models in the required region.
- **Vertex vs consumer Gemini API are different endpoints.** Same models, different auth, quotas, pricing, and sometimes feature/rollout timing. Code that works against `genai.Client()` needs `vertexai=True, project=, location=` for Vertex — and the model-name strings or available snapshots can differ slightly.
- **Model Garden models still need enabling/quota.** Calling Claude or Llama through Vertex isn't automatic — you may need to enable the partner model in Model Garden and request quota first. Expect `PERMISSION_DENIED` / `RESOURCE_EXHAUSTED` until that's done.
- **The deprecated `google-generativeai` package won't do Vertex cleanly.** Use the current **`google-genai`** SDK (with `vertexai=True`) for generation, and **`google-cloud-aiplatform`** for training/endpoints. Old `vertexai.generative_models` tutorials are being superseded by `google-genai`.
- **Quotas are project- and region-scoped and low by default.** Online-prediction QPS, GPU counts, and tokens-per-minute all have quotas you'll hit before you expect. Request increases early; don't assume the consumer API's limits apply.
- **AutoML and custom training cost real money and time.** A training job spins up managed compute billed by the minute; a runaway or misconfigured job (wrong machine type, no early stopping) is an easy way to burn budget. Start tiny.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Enterprise Gemini** (IAM, VPC-SC, CMEK, data residency, no-train guarantee) | **Vertex AI** | Same Gemini models behind Google Cloud governance. |
| **Fast prototype / hobby use** of Gemini | **Gemini API** (AI Studio key) | Instant key, free tier, no GCP project or IAM. See `google-gemini`. |
| **Deploy your own model** behind an autoscaling HTTPS endpoint | **Vertex AI Endpoints** | Managed serving, registry, monitoring — no infra to run. |
| **Many model vendors** (Gemini + Claude + Llama) under one bill/IAM | **Vertex Model Garden** | One control plane for Google, partner, and open models. |
| **Self-host open weights, no cloud lock-in** | Your own infra + vLLM/TGI | Full control, no per-token cost, but you run the ops. |

**Honest trade-offs:**

- **vs the consumer Gemini API** — identical models; Vertex adds enterprise auth/networking/compliance and the broader ML platform, at the cost of GCP setup overhead (project, billing, IAM, region). Prototype on the Gemini API, graduate to Vertex when governance demands it — the generation code barely changes.
- **vs AWS SageMaker / Azure ML** — direct analogs on the other clouds. The deciding factor is usually **which cloud you're already on** (data gravity, IAM, billing). Vertex's edge is first-party access to Gemini and a polished Model Garden.
- **vs Anthropic / OpenAI direct APIs** — going direct is simpler if you only need one vendor's models and don't need Google Cloud's governance. Vertex wins when you want *those same models* (Claude is in Model Garden) under unified Google Cloud IAM, VPC, and billing.
- **vs raw self-hosting (vLLM, Ray Serve)** — self-hosting gives maximum control and no per-token markup, but you own autoscaling, monitoring, security, and uptime. Vertex trades some cost/control for zero-ops managed serving.

**Rule of thumb:** prototype on the **Gemini API**; move to **Vertex AI** the moment you need Google Cloud governance, multi-vendor models under one roof, or managed deployment of your own models. Reach for **self-hosting** only when cost-at-scale or full control forces your hand.

## 8. Resources

- **Vertex AI documentation (official)** — https://cloud.google.com/vertex-ai/docs
- **Generative AI on Vertex AI** — https://cloud.google.com/vertex-ai/generative-ai/docs
- **Use `google-genai` with Vertex AI** — https://cloud.google.com/vertex-ai/generative-ai/docs/sdks/overview
- **`google-cloud-aiplatform` Python SDK reference** — https://cloud.google.com/python/docs/reference/aiplatform/latest
- **Model Garden (Gemini, Claude, Llama, …)** — https://cloud.google.com/model-garden
- **Vertex AI pricing** — https://cloud.google.com/vertex-ai/pricing
- **Deploy a model to an endpoint (guide)** — https://cloud.google.com/vertex-ai/docs/general/deployment

**Related notebooks:** `google-gemini` (the consumer front door to the same models), `anthropic-claude-api` (Claude is also available via Vertex Model Garden), and other entries in this domain for head-to-head comparison.